## 简介

Dropout 是在 [SRIVASTAVA2014] 中引入的一种技术，用于改善低数据条件下深度神经网络的性能，通常用于正则化。它接受一个向量作为输入，并生成相同 shape 的输出向量。

输出中的每个标量都有概率 $p$ 被设为零，否则直接从输入复制。这使得网络在仅有输入的 $1-p$ 比例的标量时也能表现良好。

在评估阶段，为了充分利用网络的能力，将 $p$ 设为 0。但是简单地将 $p$ 设为 0 会增加输出的范数，可能会人为地降低输出的 softmax temperature。为了防止这种情况发生，输出被缩放为 $\frac{1}{1-p}$，这使得无论 dropout 概率如何都能保持一致的范数。

## Baseline实现

In [ ]:
import tabulate
import torch


import triton
import triton.language as tl


@triton.jit
def _dropout(
    x_ptr,      # 输入指针
    x_keep_ptr, # pointer to a mask of 0s and 1s 由 0 和 1 组成的掩码的指针
    output_ptr, # pointer to the output 输出指针
    n_elements, # number of elements in the `x` tensor `x` 张量的元素数量
    p,          # probability that an element of `x` is changed to zero 元素 `x` 被设置为 0 的概率
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    # Load data
    # 加载数据
    x = tl.load(x_ptr + offsets, mask=mask)     #读取原始数值
    x_keep = tl.load(x_keep_ptr + offsets, mask=mask)
    # The line below is the crucial part, described in the paragraph above!
    # 下一行是上段描述的关键部分
    output = tl.where(x_keep, x / (1 - p), 0.0) #tl.where(条件, 条件为真时的值, 条件为假时的值)
    # Write-back output
    # 写回输出
    tl.store(output_ptr + offsets, output, mask=mask)


def dropout(x, x_keep, p):
    output = torch.empty_like(x)
    assert x.is_contiguous()
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    _dropout[grid](x, x_keep, output, n_elements, p, BLOCK_SIZE=1024)
    return output


# Input tensor
# 输入张量
x = torch.randn(size=(10, )).cuda()
# Dropout mask
# Dropout 掩码
p = 0.5
x_keep = (torch.rand(size=(10, )) > p).to(torch.int32).cuda()
#
output = dropout(x, x_keep=x_keep, p=p)
print(tabulate.tabulate([
    ["input"] + x.tolist(),
    ["keep mask"] + x_keep.tolist(),
    ["output"] + output.tolist(),
]))

---------  --------  --------  -------  ---------  --------  --------  --------  ---------  ------  ----------
input      0.24708   -1.13707  3.757    -0.897628  -2.70967  0.586691  0.211302  -0.167385  1.1786  -0.0643631
keep mask  1          1        1         0          0        1         0          0         1        0
output     0.494161  -2.27413  7.51399   0          0        1.17338   0          0         2.3572   0
---------  --------  --------  -------  ---------  --------  --------  --------  ---------  ------  ----------


## 第一种实现有什么问题？

这是整个教程的重点。

假设输入有 **100 万个 FP32 元素**：

```python
x.numel()  # 1_000_000
x.dtype    # torch.float32
```

输入大小为：

$$
1{,}000{,}000 \times 4 = 4\,\text{MB}
$$

如果使用教程中 `int32` 类型的 `x_keep`：

$$
\text{Mask} = 1{,}000{,}000 \times 4 = 4\,\text{MB}
$$

输出也是：

$$
\text{Output} = 4\,\text{MB}
$$

对于 Dropout 而言，Mask 的数值只有 **0 和 1**。

为了记住哪些元素被删除，我们却额外保存了一个与输入同样长的 Tensor。

尤其是在训练阶段，如果反向传播还需要使用相同的 Mask，就可能需要持续保留这部分状态。

> 那么能不能不保存 Mask，而是在需要时重新计算出来？

这就进入第二种实现。

- 第一种：提前生成 x_keep，存放在 GPU 显存中，kernel 通过 tl.load 读取。
- 低内存版本：kernel 根据 seed 和 offsets，现场生成随机数和 x_keep。

## Seeded Dropout

In [6]:
@triton.jit
def _seeded_dropout(
    x_ptr,
    output_ptr,
    n_elements,
    p,
    seed,
    BLOCK_SIZE: tl.constexpr,
):
    # compute memory offsets of elements handled by this instance
    # 计算由此实例处理的元素的内存偏移量
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    # load data from x
    # 从 x 读取数据
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    # randomly prune it
    # 随机修剪
    random = tl.rand(seed, offsets)
    x_keep = random > p
    # write-back
    # 写回
    output = tl.where(x_keep, x / (1 - p), 0.0)
    tl.store(output_ptr + offsets, output, mask=mask)


def seeded_dropout(x, p, seed):
    output = torch.empty_like(x)
    assert x.is_contiguous()
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    _seeded_dropout[grid](x, output, n_elements, p, seed, BLOCK_SIZE=1024)
    return output




x = torch.randn(size=(10, )).cuda()
# Compare this to the baseline - dropout mask is never instantiated!
# 与基线相比 - dropout 掩码从未被实例化！
output = seeded_dropout(x, p=0.5, seed=123)
output2 = seeded_dropout(x, p=0.5, seed=123)
output3 = seeded_dropout(x, p=0.5, seed=512)


print(tabulate.tabulate([
    ["input"] + x.tolist(),
    ["output (seed = 123)"] + output.tolist(),
    ["output (seed = 123)"] + output2.tolist(),
    ["output (seed = 512)"] + output3.tolist(),
]))

-------------------  ----------  ---------  --------  --------  ---------  --------  ---------  ---------  ---------  --------
input                -0.0550346  -0.269454  0.286716  -1.35822  -0.937299  0.895427  0.0653806  -0.149103  -0.877099  -2.51394
output (seed = 123)   0          -0.538908  0          0         0         1.79085   0           0         -1.7542    -5.02788
output (seed = 123)   0          -0.538908  0          0         0         1.79085   0           0         -1.7542    -5.02788
output (seed = 512)   0           0         0.573432  -2.71643   0         1.79085   0.130761    0          0          0
-------------------  ----------  ---------  --------  --------  ---------  --------  ---------  ---------  ---------  --------
